# Etapas a serem consideradas no capítulo 4

- Etapa 1 — Preparar a base de modelagem
- Etapa 2 — Criar a divisão temporal: treino, validação e teste
- Etapa 3 — Calcular baselines simples
- Etapa 4 — Avaliar os baselines com métricas
- Etapa 5 — Treinar o primeiro modelo de Machine Learning
- Etapa 6 — Comparar modelos adicionais
- Etapa 7 — Avaliar risco com modelos quantílicos e conectar com estoque de segurança

## Etapa 1 - Preparar a base de modelagem

A base deve conter:

- t_total_port_h - variável-alvo principal
- arrival_port_ts - data de referência para separação temporal
- port - porto
- region/state - localização
- operation_type - tipo de operação original
- features de calendário
- features de clima
- features de congestionamento
- flags de operação

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")

eda_file = PROJECT_ROOT / "data" / "processed" / "eda_base.parquet"

df = pd.read_parquet(eda_file)

df.shape

(129625, 126)

In [21]:
df.columns.tolist()

['port_call_id',
 'port',
 'port_name',
 'imo',
 'vessel_id',
 'vessel_name',
 'operation_type',
 'source_port',
 'source_port_name',
 'destination_port',
 'destination_port_name',
 'arrival_port_ts',
 'berthing_ts',
 'unberthing_ts',
 'departure_port_ts',
 'port_display',
 'source_port_display',
 'destination_port_display',
 'has_arrival_port_ts',
 'has_berthing_ts',
 'has_unberthing_ts',
 'has_departure_port_ts',
 'flag_arrival_after_berthing',
 'flag_berthing_after_unberthing',
 'flag_unberthing_after_departure',
 'tmp_wait_for_berthing_h',
 'tmp_operation_h',
 'tmp_post_operation_h',
 'tmp_total_port_stay_h',
 'flag_negative_tmp_wait_for_berthing_h',
 'flag_negative_tmp_operation_h',
 'flag_negative_tmp_post_operation_h',
 'flag_negative_tmp_total_port_stay_h',
 'flag_wait_too_long',
 'flag_operation_too_long',
 'flag_total_too_long',
 'flag_arrival_before_min_date',
 'eligible_for_eda',
 't_wait_for_berthing_h',
 't_operation_h',
 't_post_operation_h',
 't_total_port_stay_h',
 't_

In [22]:
df[["arrival_port_ts", "port", "operation_type", "t_total_port_stay_h"]].head()

,arrival_port_ts,port,operation_type,t_total_port_stay_h
0,2023-01-04 16:17:00,BR052001,Carga e Descarga,29.283333
1,2023-01-06 05:18:00,BR052001,Carga e Descarga,29.900000
2,2023-01-09 06:36:00,BR052001,Carga e Descarga,35.100000
3,2023-01-10 20:45:00,BR052001,Carga,31.750000
4,2023-01-17 18:46:00,BR052001,Carga e Descarga,41.150000


In [23]:
df["t_total_port_stay_h"].describe()

count    129625.000000
mean         71.103869
std          97.116162
min           0.016667
25%          18.900000
50%          38.416667
75%          80.833333
max        1280.083333
Name: t_total_port_stay_h, dtype: float64

## Etapa 2 - Limpar base e criar treino, validação e teste

O modelo deve ser treinado com observações históricas e avaliado em períodos posteriores, evitando vazamento temporal e aproximando a avaliação do cenário real de uso.

In [24]:
# Removendo linhas sem target ou sem data

model_df = df.dropna(subset=["arrival_port_ts", "t_total_port_stay_h"]).copy()

model_df["arrival_port_ts"] = pd.to_datetime(
    model_df["arrival_port_ts"],
    errors="coerce"
)

model_df = model_df.dropna(subset=["arrival_port_ts"]).copy()

model_df.shape

(129625, 126)

Divisão temporal: 

- Treino: até 2024-06-30
- Validação: 2024-07-01 até 2024-12-31
- Teste: 2025 em diante

In [25]:
model_df["arrival_port_ts"].min(), model_df["arrival_port_ts"].max()

(Timestamp('2023-01-01 00:08:00'), Timestamp('2025-12-31 23:00:00'))

In [26]:
train_df = model_df[model_df["arrival_port_ts"] < "2024-07-01"].copy()

val_df = model_df[
    (model_df["arrival_port_ts"] >= "2024-07-01")
    & (model_df["arrival_port_ts"] < "2025-01-01")
].copy()

test_df = model_df[model_df["arrival_port_ts"] >= "2025-01-01"].copy()

len(train_df), len(val_df), len(test_df)

(66909, 20362, 42354)

In [27]:
split_summary = pd.DataFrame({
    "dataset": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "min_date": [
        train_df["arrival_port_ts"].min(),
        val_df["arrival_port_ts"].min(),
        test_df["arrival_port_ts"].min(),
    ],
    "max_date": [
        train_df["arrival_port_ts"].max(),
        val_df["arrival_port_ts"].max(),
        test_df["arrival_port_ts"].max(),
    ],
})

split_summary

,dataset,rows,min_date,max_date
0,train,66909,2023-01-01 00:08:00,2024-06-30 23:30:00
1,validation,20362,2024-07-01 00:00:00,2024-12-31 22:40:00
2,test,42354,2025-01-01 00:05:00,2025-12-31 23:00:00


## Etapa 3 - Primeiro baseline: mediana global

Já iremos começar usando a mediana, e não a média, pelo fato de já termos analisado que a distribuição do tempo tem cauda longa.

In [28]:
global_median = train_df["t_total_port_stay_h"].median()

global_median

np.float64(36.9)

In [29]:
predictions = test_df[[
    "arrival_port_ts",
    "port",
    "operation_type",
    "t_total_port_stay_h",
]].copy()

predictions["pred_global_median"] = global_median

predictions.head()

,arrival_port_ts,port,operation_type,t_total_port_stay_h,pred_global_median
245,2025-01-02 02:42:00,BR052001,Carga e Descarga,27.550000,36.9
246,2025-01-04 15:18:00,BR052001,Carga,42.500000,36.9
247,2025-01-06 13:25:00,BR052001,Carga e Descarga,26.333333,36.9
248,2025-01-08 19:54:00,BR052001,Carga,26.866667,36.9
249,2025-01-11 08:00:00,BR052001,Carga,35.466667,36.9


## Etapa 4 - Métricas básicas da mediana global

Precisamos pedir o quanto a mediana está acertando nas linhas de treinamento. Vamos começar com 3 métricas:
- MAE: erro absoluto médio em horas
- RMSE: penaliza mais erros grandes
- MedAE: erro absoluto mediano

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    medae = median_absolute_error(y_true, y_pred)

    return {
        "mae": mae,
        "rmse": rmse,
        "medae": medae,
    }

calculate_metrics(
    y_true=predictions["t_total_port_stay_h"],
    y_pred=predictions["pred_global_median"],
)

{'mae': 50.006950543829,
 'rmse': np.float64(98.13395408245871),
 'medae': 23.333333333333332}

O MAE de 50,01 horas significa que, em média, esse baseline erra a permanência total em cerca de 50 horas, ou aproximadamente 2,1 dias.

O MedAE de 23,32 horas significa que, para metade dos casos, o erro absoluto fica abaixo de aproximadamente 23 horas. Isso mostra que muitos casos são relativamente bem aproximados pela mediana global.

Já o RMSE de 98,15 horas é bem maior que o MAE. Isso indica que existem erros muito grandes em algumas estadias, provavelmente por causa da cauda longa da distribuição: casos em que o navio ficou muitos dias no porto.

Essa diferença entre MAE, RMSE e MedAE é excelente para o TCC, porque mostra que o problema não é apenas prever o comportamento típico, mas também lidar com eventos extremos.

In [31]:
metrics = calculate_metrics(
    y_true=predictions["t_total_port_stay_h"],
    y_pred=predictions["pred_global_median"],
)

results = []

results.append({
    "model": "baseline_global_median",
    "dataset": "test",
    **metrics,
})

results_df = pd.DataFrame(results)

results_df

,model,dataset,mae,rmse,medae
0,baseline_global_median,test,50.006951,98.133954,23.333333


## Etapa 5 - Segundo baseline: mediana por porto

Para cada porto, calcular a mediana histórica de permanência no treino.

Depois, aplicar essa mediana aos registros de teste do mesmo porto.

In [32]:
port_median = (
    train_df
    .groupby("port")["t_total_port_stay_h"]
    .median()
    .reset_index()
    .rename(columns={"t_total_port_stay_h": "pred_median_by_port"})
)

port_median.head()

,port,pred_median_by_port
0,BR052001,26.550000
1,BR147001,36.616667
2,BR252001,70.350000
3,BR302002,52.133333
4,BR400001,34.850000


In [33]:
predictions = predictions.merge(
    port_median,
    on="port",
    how="left"
)

predictions[["port", "t_total_port_stay_h", "pred_global_median", "pred_median_by_port"]].head()

,port,t_total_port_stay_h,pred_global_median,pred_median_by_port
0,BR052001,27.550000,36.9,26.55
1,BR052001,42.500000,36.9,26.55
2,BR052001,26.333333,36.9,26.55
3,BR052001,26.866667,36.9,26.55
4,BR052001,35.466667,36.9,26.55


In [34]:
# Caso algum porto no no teste que não apareceu no treino, a previsão por porto ficará vazia. Para resolver isso, usamos a mediana global como fallback.

predictions["pred_median_by_port"] = predictions["pred_median_by_port"].fillna(global_median)

metrics = calculate_metrics(
    y_true=predictions["t_total_port_stay_h"],
    y_pred=predictions["pred_median_by_port"],
)

metrics

{'mae': 46.01086221687051,
 'rmse': np.float64(94.16143075809777),
 'medae': 20.166666666666664}

In [35]:
results.append({
    "model": "baseline_median_by_port",
    "dataset": "test",
    **metrics,
})

results_df = pd.DataFrame(results)

results_df

,model,dataset,mae,rmse,medae
0,baseline_global_median,test,50.006951,98.133954,23.333333
1,baseline_median_by_port,test,46.010862,94.161431,20.166667


Como o MAE caiu fazendo a mediana por porto, isso significa que o **porto é uma varável relevante para explicar o lead time**.

A mediana histórica por porto superou a mediana global em todas as métricas avaliadas. Esse resultado indica que a permanência portuária possui forte componente estrutural associado ao porto de operação, reforçando que políticas globais de lead time são insuficientes para representar a heterogeneidade operacional observada nos dados.

## Etapa 6 - Baseline por operação

In [36]:
model_df["operation_type"].isna().mean()

np.float64(0.00015429122468659594)

In [37]:
model_df["operation_type"].nunique()

197

In [38]:
operation_counts = (
    model_df["operation_type"]
    .value_counts(dropna=False)
    .reset_index()
)

operation_counts.columns = ["operation_type", "rows"]

operation_counts.head(20)

,operation_type,rows
0,Off-shore,33143
1,Carga,31394
2,Carga e Descarga,31174
3,Descarga,25251
4,Abastecimento (Bunker),2698
5,Desembarque/Embarque de Passageiros,1903
6,Solicitação de certificado,1293
7,Fundeio,781
8,"Abastecimento (Bunker),Carga",217
9,"Descarga,Solicitação de certificado",137


Temos 20 combinações diferentes de operações, com a maioria das ocorrências concentradas em poucas opções.

Para um baseline histórico, usamos as combinações completas para responder perguntas do tipo: Dado este porto e esta combinação histórica de operações, qual foi a mediana de permanência observada no passado?

In [ ]:
# Verificando se as combinações são esparsas:

port_operation_counts = (
    train_df
    .groupby(["port", "operation_type"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

port_operation_counts.head(20)

,port,operation_type,rows
417,BRRIO,Off-shore,7916
431,BRRJ022,Off-shore,6733
525,BRSSZ,Carga e Descarga,3239
524,BRSSZ,Carga,3158
272,BRMEA001,Off-shore,1621
329,BRPNG,Carga e Descarga,1619
402,BRRIO,Carga e Descarga,1521
528,BRSSZ,Descarga,1288
328,BRPNG,Carga,1086
73,BRARE,Descarga,981


In [40]:
port_operation_counts["rows"].describe()

count     636.000000
mean      105.202830
std       484.760294
min         1.000000
25%         1.000000
50%         3.000000
75%        35.250000
max      7916.000000
Name: rows, dtype: float64

In [ ]:
# Qual a proporção de combinações de porto e tipo de operação que tem menos de 30 exemplos no treino?
# Se esse número for alto, significa que muitos grupos são pequenos. Isso não impede o baseline, mas exige cuidado na interpretação dos resultados.

(port_operation_counts["rows"] < 30).mean()

np.float64(0.7342767295597484)

Agora criando o baseline por combo de operação:

In [47]:
port_operation_median = (
    train_df
    .groupby(["port", "operation_type"], dropna=False)["t_total_port_stay_h"]
    .median()
    .reset_index()
    .rename(columns={"t_total_port_stay_h": "pred_median_by_port_operation_combo"})
)

port_operation_median.head()

,port,operation_type,pred_median_by_port_operation_combo
0,BR052001,Carga,26.800000
1,BR052001,Carga e Descarga,26.416667
2,BR052001,Off-shore,342.000000
3,BR147001,Carga,56.950000
4,BR147001,Carga e Descarga,24.750000


In [48]:
predictions = predictions.merge(
    port_operation_median,
    on=["port", "operation_type"],
    how="left"
)

# Como pode existir combinação no teste que não apareceu no treino, usamos fallback para a mediana por porto:
predictions["pred_median_by_port_operation_combo"] = (
    predictions["pred_median_by_port_operation_combo"]
    .fillna(predictions["pred_median_by_port"])
    .fillna(global_median)
)

In [49]:
metrics = calculate_metrics(
    y_true=predictions["t_total_port_stay_h"],
    y_pred=predictions["pred_median_by_port_operation_combo"],
)

metrics

{'mae': 42.69011033983409,
 'rmse': np.float64(88.69576758664962),
 'medae': 17.166666666666664}

In [50]:
results.append({
    "model": "baseline_median_by_port_operation_combo",
    "dataset": "test",
    **metrics,
})

results_df = pd.DataFrame(results)

results_df

,model,dataset,mae,rmse,medae
0,baseline_global_median,test,50.006951,98.133954,23.333333
1,baseline_median_by_port,test,46.010862,94.161431,20.166667
2,baseline_median_by_port_operation_combo,test,42.690110,88.695768,17.166667


O tipo de operação, mesmo representado como combinação, adiciona informação preditiva além do porto.

In [51]:
# Criando uma coluna de melhoria percentual
baseline_mae = results_df.loc[
    results_df["model"] == "baseline_global_median",
    "mae"
].iloc[0]

results_df["mae_improvement_vs_global_median_pct"] = (
    (baseline_mae - results_df["mae"]) / baseline_mae * 100
)

results_df

,model,dataset,mae,rmse,medae,mae_improvement_vs_global_median_pct
0,baseline_global_median,test,50.006951,98.133954,23.333333,0.000000
1,baseline_median_by_port,test,46.010862,94.161431,20.166667,7.991066
2,baseline_median_by_port_operation_combo,test,42.690110,88.695768,17.166667,14.631646


In [52]:
output_dir = PROJECT_ROOT / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "chapter4_baseline_results.csv",
    index=False,
)